In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
import warnings
warnings.filterwarnings("ignore")

print("OK")

OK


In [2]:
# 20 empresas DAX seleccionadas — representativas de cada sector
DAX_COMPANIES = {
    # Industrials
    "Siemens":          {"ticker": "SIE.DE",  "sector": "Industrials"},
    "Airbus":           {"ticker": "AIR.DE",  "sector": "Industrials"},
    "Infineon":         {"ticker": "IFX.DE",  "sector": "Industrials"},
    # Chemicals
    "BASF":             {"ticker": "BAS.DE",  "sector": "Chemicals"},
    "Bayer":            {"ticker": "BAYN.DE", "sector": "Chemicals"},
    "Merck":            {"ticker": "MRK.DE",  "sector": "Chemicals"},
    # Automotive
    "Volkswagen":       {"ticker": "VOW3.DE", "sector": "Automotive"},
    "BMW":              {"ticker": "BMW.DE",  "sector": "Automotive"},
    "Mercedes-Benz":    {"ticker": "MBG.DE",  "sector": "Automotive"},
    # Financials
    "Deutsche Bank":    {"ticker": "DBK.DE",  "sector": "Financials"},
    "Allianz":          {"ticker": "ALV.DE",  "sector": "Financials"},
    "Munich Re":        {"ticker": "MUV2.DE", "sector": "Financials"},
    # Technology
    "SAP":              {"ticker": "SAP.DE",  "sector": "Technology"},
    "Deutsche Telekom": {"ticker": "DTE.DE",  "sector": "Technology"},
    # Consumer
    "Adidas":           {"ticker": "ADS.DE",  "sector": "Consumer"},
    "Henkel":           {"ticker": "HEN3.DE", "sector": "Consumer"},
    "Beiersdorf":       {"ticker": "BEI.DE",  "sector": "Consumer"},
    # Healthcare
    "Fresenius":        {"ticker": "FRE.DE",  "sector": "Healthcare"},
    "Sartorius":        {"ticker": "SRT3.DE", "sector": "Healthcare"},
    # Logistics
    "Deutsche Post":    {"ticker": "DHL.DE",  "sector": "Logistics"},
}

print(f"Empresas definidas: {len(DAX_COMPANIES)}")
for sector in set(v["sector"] for v in DAX_COMPANIES.values()):
    count = sum(1 for v in DAX_COMPANIES.values() if v["sector"] == sector)
    print(f"  {sector}: {count} empresas")

Empresas definidas: 20
  Technology: 2 empresas
  Industrials: 3 empresas
  Automotive: 3 empresas
  Chemicals: 3 empresas
  Financials: 3 empresas
  Logistics: 1 empresas
  Healthcare: 2 empresas
  Consumer: 3 empresas


In [3]:
YEAR_START = "2019-01-01"
YEAR_END   = "2024-01-01"

prices_list = []

for name, info in DAX_COMPANIES.items():
    print(f"Descargando precios: {name}...")
    df = yf.download(info["ticker"], start=YEAR_START, end=YEAR_END,
                     auto_adjust=True, progress=False)

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    temp = df[["Close"]].copy().reset_index()
    temp.columns = ["date", "close"]
    temp["company"] = name
    temp["sector"]  = info["sector"]
    temp["ticker"]  = info["ticker"]
    prices_list.append(temp)

prices = pd.concat(prices_list, ignore_index=True)
print(f"\nPrecios descargados: {len(prices)} filas")
print(prices.groupby("sector")["close"].count())

Descargando precios: Siemens...
Descargando precios: Airbus...
Descargando precios: Infineon...
Descargando precios: BASF...
Descargando precios: Bayer...
Descargando precios: Merck...
Descargando precios: Volkswagen...
Descargando precios: BMW...
Descargando precios: Mercedes-Benz...
Descargando precios: Deutsche Bank...
Descargando precios: Allianz...
Descargando precios: Munich Re...
Descargando precios: SAP...
Descargando precios: Deutsche Telekom...
Descargando precios: Adidas...
Descargando precios: Henkel...
Descargando precios: Beiersdorf...
Descargando precios: Fresenius...
Descargando precios: Sartorius...
Descargando precios: Deutsche Post...

Precios descargados: 25440 filas
sector
Automotive     3816
Chemicals      3816
Consumer       3816
Financials     3816
Healthcare     2544
Industrials    3816
Logistics      1272
Technology     2544
Name: close, dtype: int64


In [4]:
financials_list = []

for name, info in DAX_COMPANIES.items():
    print(f"Descargando financials: {name}...")
    stock = yf.Ticker(info["ticker"])
    i     = stock.info

    financials_list.append({
        "company":        name,
        "sector":         info["sector"],
        "ticker":         info["ticker"],
        "market_cap":     i.get("marketCap",           None),
        "enterprise_value":i.get("enterpriseValue",    None),
        "revenue_ttm":    i.get("totalRevenue",         None),
        "ebitda":         i.get("ebitda",               None),
        "ev_revenue":     i.get("enterpriseToRevenue",  None),
        "ev_ebitda":      i.get("enterpriseToEbitda",   None),
        "pe_ratio":       i.get("trailingPE",           None),
        "gross_margins":  i.get("grossMargins",         None),
        "revenue_growth": i.get("revenueGrowth",        None),
        "profit_margins": i.get("profitMargins",        None),
        "return_on_equity":i.get("returnOnEquity",      None),
        "debt_to_equity": i.get("debtToEquity",         None),
    })

fin_df = pd.DataFrame(financials_list)

print("\nMétricas descargadas:")
print(fin_df[["company", "sector", "ev_ebitda",
              "gross_margins", "revenue_growth"]].to_string(index=False))

Descargando financials: Siemens...
Descargando financials: Airbus...
Descargando financials: Infineon...
Descargando financials: BASF...
Descargando financials: Bayer...
Descargando financials: Merck...
Descargando financials: Volkswagen...
Descargando financials: BMW...
Descargando financials: Mercedes-Benz...
Descargando financials: Deutsche Bank...
Descargando financials: Allianz...
Descargando financials: Munich Re...
Descargando financials: SAP...
Descargando financials: Deutsche Telekom...
Descargando financials: Adidas...
Descargando financials: Henkel...
Descargando financials: Beiersdorf...
Descargando financials: Fresenius...
Descargando financials: Sartorius...
Descargando financials: Deutsche Post...

Métricas descargadas:
         company      sector  ev_ebitda  gross_margins  revenue_growth
         Siemens Industrials     23.039        0.38845          -0.000
          Airbus Industrials     21.057        0.15366          -0.066
        Infineon Industrials     26.561   

In [5]:
prices.to_csv("../data/dax_prices.csv", index=False)
fin_df.to_csv("../data/dax_financials.csv", index=False)

print("Guardado OK")
print(f"\nNulos por columna en financials:")
print(fin_df.isnull().sum())

Guardado OK

Nulos por columna en financials:
company             0
sector              0
ticker              0
market_cap          0
enterprise_value    0
revenue_ttm         0
ebitda              2
ev_revenue          0
ev_ebitda           2
pe_ratio            1
gross_margins       0
revenue_growth      1
profit_margins      0
return_on_equity    0
debt_to_equity      2
dtype: int64
